# 04 - Detection & classification benchmark

Companion notebook to `results/runs.jsonl` and `results/figures/bench_*.png`.
Event-level detection (no point-adjusted F1) on the temporal test split and the
household hold-out, plus the supervised type/origin classifier.

Prerequisites: build the benchmark first, e.g.
`python -m energymind.benchmark --prevalence 0.05 --seeds 1 2 3`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from energymind import benchmark_plots as B, io
from energymind.config import load_settings

warnings.filterwarnings("ignore", category=FutureWarning)
%config InlineBackend.figure_format = "png"
plt.rcParams["figure.dpi"] = 110

PREF, SEED = 0.05, 1
settings = load_settings()
runs = B.load_runs()
FIG = io.resolve("results/figures")
print(f"{len(runs)} runs | {runs['model'].nunique()} models | seeds {sorted(runs['seed'].unique())}")

## Detection at a glance

Event-F1 at the fixed false-alarm budget (1 per household-week), event-level
threshold selection on validation, mean over seeds. `.oracle` is the constrained-free
ceiling for the same score.

In [ ]:
det = runs[(runs.task == "detection") & (runs.split == "test")].copy()
table = (
    det.groupby(["model", "thresholding"])
    .agg(F1=("event_f1", "mean"), sd=("event_f1", "std"), recall=("event_recall", "mean"),
         precision=("alert_precision", "mean"), pr_auc=("pr_auc", "mean"),
         fa=("fa_per_household_week", "mean"))
    .round(3)
    .sort_values("F1", ascending=False)
)
table.style.hide(axis="index")

## Classification (type + origin)

LightGBM vs Random Forest, isotonic-calibrated, on train events; chosen by
macro-F1 on validation; reported on test.

In [ ]:
clf = runs[runs.task == "classification"].copy()
summary = (
    clf.groupby("model")[["type_macro_f1", "origin_accuracy", "origin_ece"]]
    .agg(["mean", "std"])
    .round(3)
)
summary.style.hide(axis="index")

## Leaderboard and trade-off

In [ ]:
B.fig_detection_leaderboard(runs, FIG)
plt.show()

In [ ]:
B.fig_detector_tradeoff(runs, FIG)
plt.show()
B.fig_recall_breakdown(runs, FIG)
plt.show()

## Representative run (score distributions, PR curves, classifier)

Computed once for prevalence 5%, seed 1 (the LSTM-AE row is taken from
`runs.jsonl` to keep the notebook quick).

In [ ]:
data = B.prepare_run(settings, prevalence=PREF, seed=SEED)
print("detectors:", list(data["scores"]))
print("events:", len(data["events"].anomaly_id.unique()))

In [ ]:
B.fig_score_distributions(data["frame"], data["scores"], data["events"], FIG)
plt.show()
B.fig_pr_curves(data["frame"], data["scores"], data["events"], FIG)
plt.show()

In [ ]:
if len(data["classes"]):
    B.fig_classifier(runs, data["confusion"], data["classes"], data["reliability"], FIG)
    plt.show()

## Notes

* Detection is **event-level** (`event_f1`); precision is measured over alerts,
  so no point-adjusted F1 is used.
* `rules` leads the unsupervised set; the supervised model is the labelled
  upper bound and is reported separately; the ensemble improves PR-AUC.
* `recall_by_variant` and `recall_by_length` (in `runs.jsonl`) localise the
  misses: naive > consistent, and short events are the hardest.